# Road Only

Day 18 · NULL. US counties with zero public transport: no fixed-route bus or rail stop, no Amtrak stop, no intercity
bus or rail terminal, no ferry. The only way in or out is a road. Run `python scripts\fetch_week3.py day18` first.

## 1 · Settings

In [ ]:
DAY      = 18
PROMPT   = "NULL"
SLUG     = "null-road-only-counties"
NOTEBOOK = "day18_null_road_only_counties.ipynb"
TITLE    = "Road Only"
CREDITS  = ("Transit stops: BTS National Transit Map (NTAD). Amtrak stations and routes: BTS NTAD. Intercity bus, rail "
            "and ferry terminals: BTS Intermodal Passenger Connectivity Database. Counties: US Census Bureau.")

COUNT_AIRPORTS = False     # True = a county with a commercial airport counts as connected

PAGE, INK, SOFT = "#efede8", "#1d1d1f", "#7a7a76"
ROAD, LINKED = "#2f3033", "#d9d6cf"        # asphalt for road-only, pale for connected
RAIL = "#e0442b"                            # Amtrak, signal red
YELLOW = "#f2c230"                          # road-sign yellow, for the counties the train passes without stopping

## 2 · Load

In [ ]:
import numpy as np, pandas as pd, geopandas as gpd, matplotlib.pyplot as plt
from mapkit import poster, publish
from mapkit.paths import PROCESSED, EQUAL_AREA

counties = gpd.read_file(PROCESSED / "counties_connectivity.geojson").to_crs(EQUAL_AREA)
routes = gpd.read_file(PROCESSED / "amtrak_routes.geojson").to_crs(EQUAL_AREA)
stations = gpd.read_file(PROCESSED / "amtrak_stations.geojson").to_crs(EQUAL_AREA)
print(f"{len(counties):,} counties · {len(stations):,} Amtrak stops · {len(routes)} Amtrak route pieces")

## 3 · Analysis, every number on the map comes from here

In [ ]:
links = {
    "fixed-route transit": counties.transit_stops > 0,
    "Amtrak stop":         counties.amtrak_stations > 0,
    "intercity bus":       counties.ipcd_bus_i > 0,
    "rail terminal":       counties.ipcd_mode_rail > 0,
    "ferry":               counties.ipcd_mode_ferry > 0,
}
if COUNT_AIRPORTS:
    links["airport"] = counties.ipcd_mode_air > 0
connected = np.logical_or.reduce(list(links.values()))
counties["road_only"] = ~connected

# Counties the train passes through without a single stop of any kind
rail_stops = stations[stations.StnType.str.upper().str.contains("RAIL", na=False)]
crossed = gpd.sjoin(counties[["geoid", "geometry"]], routes[["geometry"]], predicate="intersects").geoid.unique()
counties["train_passes"] = counties.geoid.isin(crossed)
counties["passed_by"] = counties.road_only & counties.train_passes

area = counties.area
N = len(counties)
STATS = dict(road=int(counties.road_only.sum()), passed=int(counties.passed_by.sum()),
             area_share=area[counties.road_only].sum() / area.sum() * 100)
STATS["share"] = STATS["road"] / N * 100
print(f"{STATS['road']:,} of {N:,} counties in the Lower 48 ({STATS['share']:.0f}%) are road only")
print(f"They cover {STATS['area_share']:.0f}% of the land")
print(f"{STATS['passed']} of them have an Amtrak line running through and nowhere to get on")
print("Counties connected by each kind of link:", {k: int(v.sum()) for k, v in links.items()})
print(counties[counties.road_only].state_abb.value_counts().head(8).to_dict())

## 4 · Map

In [ ]:
F = poster.font
TITLE_F, TEXT_F, TEXT_B = poster.SLAB, poster.BARLOW, poster.BARLOW_BOLD     # Zilla Slab, like a road sign
fig, ax = poster.page(PAGE)
ax.set_position([0.02, 0.19, 0.96, 0.60])
counties[~counties.road_only].plot(ax=ax, color=LINKED, edgecolor=PAGE, lw=0.15, zorder=1)
counties[counties.road_only & ~counties.passed_by].plot(ax=ax, color=ROAD, edgecolor=PAGE, lw=0.15, zorder=2)
counties[counties.passed_by].plot(ax=ax, color=YELLOW, edgecolor=PAGE, lw=0.15, zorder=3)
routes.plot(ax=ax, color=RAIL, lw=1.1, zorder=4)
ax.scatter(rail_stops.geometry.x, rail_stops.geometry.y, s=5, color=PAGE, edgecolor=RAIL, lw=0.6, zorder=5)
b = counties.total_bounds
ax.set_xlim(b[0], b[2]); ax.set_ylim(b[1], b[3]); ax.set_aspect("equal"); ax.set_axis_off()

fig.text(0.06, 0.955, TITLE.upper(), color=INK, fontproperties=F(TITLE_F, 76), va="top")
fig.text(0.06, 0.858, f"{STATS['road']:,} US counties have no bus, no train, no ferry. Just the road.",
         color=INK, fontproperties=F(TEXT_F, 17), va="top")
fig.text(0.06, 0.828, f"In {STATS['passed']} of them, an Amtrak line runs straight through and never stops.",
         color=SOFT, fontproperties=F(TEXT_F, 15), va="top")

# legend
lx, ly = 0.06, 0.155
for i, (col, lab, line) in enumerate([(ROAD, "Road only", False), (YELLOW, "Road only, but a train passes through", False),
                                      (LINKED, "Some public transport", False), (RAIL, "Amtrak line and station", True)]):
    y = ly - i * 0.026
    if line:
        fig.add_artist(plt.Line2D([lx, lx + 0.03], [y + 0.006] * 2, transform=fig.transFigure, color=col, lw=2))
    else:
        fig.add_artist(plt.Rectangle((lx, y), 0.03, 0.013, transform=fig.transFigure, color=col, lw=0))
    fig.text(lx + 0.042, y + 0.0065, lab, color=INK, fontproperties=F(TEXT_F, 12.5), va="center")

poster.credit(fig, "BTS National Transportation Atlas Database:\nNational Transit Map, Amtrak, Intermodal Connectivity\n"
                   "US Census counties · Lower 48 · dial-a-ride and airports not counted",
              x=0.50, y=0.065, color=SOFT, path=TEXT_F, name_path=TEXT_B, name_color=INK, size=9)
plt.show()

## 5 · Export

In [ ]:
publish.save(fig, day=DAY, slug=SLUG, prompt=PROMPT, title=TITLE,
             subtitle=f"{STATS['road']:,} US counties with no public transport; in {STATS['passed']}, Amtrak runs through without stopping",
             series="city", tool="Python", notebook=NOTEBOOK, credits=CREDITS)